# Ground Truth C-Space Visualization (Interactive)
This notebook provides an interactive 3D visualization of the **Ground Truth** Configuration Space generated by the MuJoCo oracle.

In [6]:
import os
import numpy as np
import plotly.graph_objects as go
from skimage import measure
import plotly.io as pio
pio.renderers.default = 'notebook_connected'


In [7]:
grid_path = "output/ndf/jaco_cspace.npy"
theta0_path = "output/ndf/jaco_theta0.npy"
theta1_path = "output/ndf/jaco_theta1.npy"
theta2_path = "output/ndf/jaco_theta2.npy"

if all(os.path.exists(p) for p in [grid_path, theta0_path, theta1_path, theta2_path]):
    grid = np.load(grid_path)
    t0 = np.load(theta0_path)
    t1 = np.load(theta1_path)
    t2 = np.load(theta2_path)
    ticks = [t0, t1, t2]
    print(f"Loaded grid with shape {grid.shape}")
else:
    print("Error: Ground truth data not found. Please run generate_true_cspace.py first.")

Loaded grid with shape (64, 64, 64)


In [8]:
print("Extracting isosurface...")
# iso=0.5 because grid is 1 (obs) and 0 (free)
verts, faces, normals, values = measure.marching_cubes(grid, level=0.5)

for i in range(3):
    verts[:, i] = np.interp(verts[:, i], np.arange(len(ticks[i])), ticks[i])

print(f"Extracted mesh with {len(verts)} vertices and {len(faces)} faces.")

Extracting isosurface...
Extracted mesh with 20151 vertices and 40298 faces.


In [11]:
# Using Unicode symbols (θ, π) for maximum compatibility across all notebook environments
fig = go.Figure(data=[
    go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color='blue', opacity=0.6, name='Ground Truth'
    )
])

fig.update_layout(
    title="Ground Truth C-Space Mesh (Interactive)",
    # template="plotly_dark",
    scene=dict(
        xaxis=dict(
            title='θ₁',
            tickvals=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
            ticktext=['-π', '-π/2', '0', 'π/2', 'π']
        ),
        yaxis=dict(
            title='θ₂',
            tickvals=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
            ticktext=['-π', '-π/2', '0', 'π/2', 'π']
        ),
        zaxis=dict(
            title='θ₃',
            tickvals=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
            ticktext=['-π', '-π/2', '0', 'π/2', 'π']
        ),
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()